In [1]:
import sagemaker
import boto3 # python sdk for AWS

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sess=sagemaker.Session() # it gets session of the the sagemaker notebook
sess

In [3]:
role=sagemaker.get_execution_role() # Role of the sagemaker notebook basically an IAM role for the notebook
role

'<SAGEMAKER_EXECUTION_ROLE_ARN>'

In [4]:
sess.boto_region_name

'us-east-2'

In [5]:

region_mapping = {
    "af-south-1": "626614931356",
    "il-central-1": "780543022126",
    "ap-east-1": "871362719292",
    "ap-northeast-1": "763104351884",
    "ap-northeast-2": "763104351884",
    "ap-northeast-3": "364406365360",
    "ap-south-1": "763104351884",
    "ap-south-2": "772153158452",
    "ap-southeast-1": "763104351884",
    "ap-southeast-2": "763104351884",
    "ap-southeast-3": "907027046896",
    "ap-southeast-4": "457447274322",
    "ca-central-1": "763104351884",
    "cn-north-1": "727897471807",
    "cn-northwest-1": "727897471807",
    "eu-central-1": "763104351884",
    "eu-central-2": "380420809688",
    "eu-north-1": "763104351884",
    "eu-west-1": "763104351884",
    "eu-west-2": "763104351884",
    "eu-west-3": "763104351884",
    "eu-south-1": "692866216735",
    "eu-south-2": "503227376785",
    "me-south-1": "217643126080",
    "me-central-1": "914824155844",
    "sa-east-1": "763104351884",
    "us-east-1": "763104351884",
    "us-east-2": "763104351884",
    "us-gov-east-1": "446045086412",
    "us-gov-west-1": "442386744353",
    "us-iso-east-1": "886529160074",
    "us-isob-east-1": "094389454867",
    "us-west-1": "763104351884",
    "us-west-2": "763104351884",
}

llm_image = f"{region_mapping[sess.boto_region_name]}.dkr.ecr.{sess.boto_region_name}.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi1.3.3-gpu-py310-cu121-ubuntu20.04-v1.0"

print(f"llm image uri: {llm_image}")

llm image uri: 763104351884.dkr.ecr.us-east-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi1.3.3-gpu-py310-cu121-ubuntu20.04-v1.0


In [6]:
s3_uri = "s3://<SAGEMAKER_DEFAULT_BUCKET>/mixtral-8x7b-qlora-2026-08-12-21-08-11-2026-08-12-21-08-11-975/output/model.tar.gz"
print(s3_uri)

s3://<SAGEMAKER_DEFAULT_BUCKET>/mixtral-8x7b-qlora-2026-08-12-21-08-11-2026-08-12-21-08-11-975/output/model.tar.gz


In [9]:
import json
from sagemaker.huggingface import HuggingFaceModel


instance_type = "ml.g5.48xlarge"
number_of_gpu = 8
health_check_timeout = 300

config = {
    "HF_MODEL_ID": "/opt/ml/model",
    "SM_NUM_GPUS": json.dumps(number_of_gpu),
    "MAX_INPUT_LENGTH": json.dumps(24000), 
    "MAX_BATCH_PREFILL_TOKENS": json.dumps(32000),
    "MAX_TOTAL_TOKENS": json.dumps(32000),
    "MAX_BATCH_TOTAL_TOKENS": json.dumps(
        512000
    ),
}


llm_model = HuggingFaceModel(model_data=s3_uri, role=role, image_uri=llm_image, env=config)

In [10]:
endpoint_name = sagemaker.utils.name_from_base("Mixtral-8x7B-endpoint-v1")

llm = llm_model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type=instance_type,
    container_startup_health_check_timeout=health_check_timeout,
)

-------------------------------------------------------------------------!

In [3]:
from sagemaker.huggingface import HuggingFacePredictor

endpoint_name = "Mixtral-8x7B-endpoint-v1-2026-08-19-05-17-44-104"

llm = HuggingFacePredictor(
    endpoint_name=endpoint_name
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
prompt = "What is Amazon SageMaker?"


payload = {
    "do_sample": True,          # Enable sampling instead of always choosing
                                # the single most likely next token

    "top_p": 0.6,               # Nucleus sampling:
                                # keep the smallest set of likely tokens whose
                                # cumulative probability reaches 60%

    "temperature": 0.1,         # Controls randomness
                                # Low temperature = more predictable/deterministic output

    "top_k": 50,                # Consider only the top 50 most likely tokens
                                # when generating the next token

    "max_new_tokens": 1024,     # Maximum number of NEW tokens the model can generate

    "repetition_penalty": 1.03, # Slightly discourage repeating the same
                                # words/tokens/phrases

    "return_full_text": False,  # Return only the newly generated text,
                                # not the original input prompt

    "stop": ["</s>"],           # Stop generation when this end-of-sequence
                                # marker is encountered
}

In [5]:
chat = llm.predict({"inputs": prompt, "parameters": payload})

print(chat[0]["generated_text"])



Amazon SageMaker is a fully managed service that provides every developer and data scientist with the ability to build, train, and deploy machine learning (ML) models quickly. SageMaker removes the heavy lifting from each step of the machine learning process to make it easier to develop high quality models.

What are the key benefits of using Amazon SageMaker?

Amazon SageMaker removes the heavy lifting from each step of the machine learning process to make it easier to develop high quality models.

Build faster with optimized algorithms and pre-built templates
SageMaker includes 15+ optimized algorithms for training and inference such as XGBoost, Linear Learner, Factorization Machines, and others. These algorithms are optimized for training on large datasets and deployment on a wide variety of instances. SageMaker also provides built-in support for all popular open source deep learning frameworks like TensorFlow, Apache MXNet, PyTorch, Chainer, Keras, and Gluon. You can use these al